Community Assistanty to allow Backed by a Firebase Database

Business Justification: I have a community management solution built by me (https://audiatur.co). In simple terms, Audiatur provides all the tools(organising elections, conference management, event calendar, general info storage) needed to manage a community(a community could be a church, association etc).

One key requirement is to allow community members get info about the community using Whatsapp. This project makes that possible. Data is stored on Firebase

First the imports

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
import firebase_admin
from firebase_admin import credentials, firestore
import json
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
import gradio as gr

Lets setup model

In [4]:

MODEL = "gpt-4.1-nano"
db_name = "vector_db"
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key exists and begins sk-proj-


now lets setup Firebase so we can load data

In [5]:
config = json.loads(os.getenv("FIREBASE_CONFIG_JSON"))
cred = credentials.Certificate(config)
try:
    firebase_admin.initialize_app(cred)
    print("Firebase initialized successfully!")
except ValueError:
    print("App already initialized.")



Firebase initialized successfully!


Load communities and conferences

In [6]:
db = firestore.client()
projects = db.collection('projects').get()
for project in projects:
    print(project.to_dict())


{'userId': '', 'status': 'INITIATED', 'shortDescription': '', 'id': 'f618e67b-baa0-42af-9386-bc4e0c63dd11', 'endDate': DatetimeWithNanoseconds(2025, 9, 22, 19, 57, 47, 320000, tzinfo=datetime.timezone.utc), 'name': 'First Game', 'startDate': DatetimeWithNanoseconds(2025, 9, 22, 19, 57, 47, 320000, tzinfo=datetime.timezone.utc), 'description': '', 'createdById': '', 'projectType': 'GAME', 'createdAt': DatetimeWithNanoseconds(2025, 9, 22, 19, 57, 47, 320000, tzinfo=datetime.timezone.utc)}
{'userId': '', 'status': 'INITIATED', 'questionSource': 'AIGENERATED', 'shortDescription': 'This is game 3, a game to test full implementation', 'id': 'd4a6f319-3faf-44ee-aeb1-b3402f67d59f', 'endDate': DatetimeWithNanoseconds(2025, 11, 12, 5, 53, 7, 56000, tzinfo=datetime.timezone.utc), 'passedDomain': 'game3', 'name': 'Game 3', 'startDate': DatetimeWithNanoseconds(2025, 11, 12, 5, 53, 7, 56000, tzinfo=datetime.timezone.utc), 'description': '', 'domains': ['game3.audiatur.co'], 'createdById': '', 'creat

Lets create Langchain documents

In [8]:
documents = [Document(page_content=project.to_dict()['description'], metadata={'source': project.id}) for project in projects]
print(documents[2])

page_content='This is the very first election' metadata={'source': 'IQzLjIll0P6GqCZaOvBz'}


Lets chunk the data

In [9]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)
print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 4 chunks
First chunk:

page_content='This is the very first election' metadata={'source': 'IQzLjIll0P6GqCZaOvBz'}


Lets create our vector and vector db

In [10]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
print(embeddings)
if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")


client=<openai.resources.embeddings.Embeddings object at 0x12ab666f0> async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x12aaa39e0> model='text-embedding-3-large' dimensions=None deployment='text-embedding-ada-002' openai_api_version=None openai_api_base=None openai_api_type=None openai_proxy=None embedding_ctx_length=8191 openai_api_key=SecretStr('**********') openai_organization=None allowed_special=None disallowed_special=None chunk_size=1000 max_retries=2 request_timeout=None headers=None tiktoken_enabled=True tiktoken_model_name=None show_progress_bar=False model_kwargs={} skip_empty=False default_headers=None default_query=None retry_min_seconds=4 retry_max_seconds=20 http_client=None http_async_client=None check_embedding_ctx_length=True
Vectorstore created with 4 documents


Lets start retrieving

In [ ]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant that helps users get information about a community.
You are chatting with a user about a community.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""


Lets setup chat function

In [13]:
from langchain_core.messages import HumanMessage, SystemMessage


def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

Lets wire up Gradio

In [ ]:
gr.ChatInterface(answer_question).launch()